# Evaluate best checkpoint with single-view eval: in-sample + OOF, LR + LDA

This notebook uses the **training-time encoder definition** copied from the original training notebook, and evaluates one fixed checkpoint with a **single-view eval dataset** so each file contributes its true windows once.

It reports frog-level metrics for:

- **in-sample LR**
- **in-sample LDA**
- **OOF LOGO LR**
- **OOF LOGO LDA**

Aggregation is done at frog level using file probabilities (`mean` or `median`).


In [ ]:

# =========================================================
# 0) Imports
# =========================================================
import re
import math
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import roc_auc_score, average_precision_score, accuracy_score
from sklearn.metrics import balanced_accuracy_score
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis




print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())


In [ ]:
# =========================================================
# 1) Config

# =========================================================
PROJECT_ROOT = Path("../..").resolve()
NPZ_DIR=PROJECT_ROOT / "data" / "processed" /"mzML_npz_45"
META_CSV =PROJECT_ROOT / "data" / "processed" / "metadata_with_frog.csv"
CKPT_DIR =PROJECT_ROOT /"results"/"final"/"Task1710"/"1710_BestCKPT"
OUTPUT_DIR = PROJECT_ROOT /"results"/"runs"/"Probe_compare"/"Task1710"/"logs1"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

WINDOW_SIZE = 500
STRIDE = 250
JITTER_MAX = 0

EMBED_DIM = 64
PATCH_SIZE = 50
ENCODER_STRIDE = 50
NUM_HEADS = 4

LOGREG_C = 40.0
FROG_PROB_AGG = "mean"   # "mean" or "median"

POS_LABEL = "STP1710.7"
NEG_LABEL = "control"

LDA_SHRINKAGE = "auto"


In [ ]:

# =========================================================
# 2) Exact encoder definition from the training notebook
# =========================================================
class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model):
        super().__init__()
        self.d_model = d_model
        div_term = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        self.register_buffer("div_term", div_term)

    def forward(self, rt):
        rt = rt.unsqueeze(-1)  # (B,seq,1)
        pe = torch.zeros(rt.size(0), rt.size(1), self.d_model, device=rt.device)
        pe[:, :, 0::2] = torch.sin(rt * self.div_term)
        pe[:, :, 1::2] = torch.cos(rt * self.div_term)
        return pe


class MassSpecWindowContrastEncoder(nn.Module):
    """
    Window-level encoder:
    Patch(Conv1d) + Transformer + CLS + RT sinusoidal positional encoding.
    Input:  signal(B,L), rt(B,L)
    Output: embedding(B,embed_dim)
    """
    def __init__(self, patch_size=50, stride=50, embed_dim=64, num_heads=4):
        super().__init__()
        self.patch_size = patch_size
        self.stride = stride
        self.embed_dim = embed_dim

        self.conv = nn.Conv1d(
            in_channels=1,
            out_channels=embed_dim,
            kernel_size=patch_size,
            stride=stride,
            padding=0,
        )

        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.positional_encoding = SinusoidalPositionalEncoding(embed_dim)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=4 * embed_dim,
            batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=1)

    def forward(self, signal, rt):
        B = signal.size(0)

        if not torch.isfinite(signal).all():
            raise RuntimeError("signal contains NaN/Inf before encoder")
        if not torch.isfinite(rt).all():
            raise RuntimeError("rt contains NaN/Inf before encoder")

        # light numeric stabilization: keep shape, only shrink magnitude
        signal = signal / 1e6

        x = self.conv(signal.unsqueeze(1))   # (B, embed_dim, n_patches)
        if not torch.isfinite(x).all():
            raise RuntimeError("conv output contains NaN/Inf")

        x = x.permute(0, 2, 1)               # (B, n_patches, embed_dim)

        rt_patch = rt[:, self.patch_size - 1::self.stride]
        assert x.size(1) == rt_patch.size(1), (
            f"Patch/RT mismatch: conv patches={x.size(1)} vs rt patches={rt_patch.size(1)}. "
            f"patch_size={self.patch_size}, stride={self.stride}, input_len={rt.size(1)}"
        )

        rt_patch = rt_patch / 1800.0
        pe = self.positional_encoding(rt_patch)
        if not torch.isfinite(pe).all():
            raise RuntimeError("positional encoding contains NaN/Inf")

        x = x + pe
        if not torch.isfinite(x).all():
            raise RuntimeError("x + positional encoding contains NaN/Inf")

        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)

        x = self.transformer(x)
        if not torch.isfinite(x).all():
            raise RuntimeError("transformer output contains NaN/Inf")

        h = x[:, 0, :]
        h = F.layer_norm(h, h.shape[-1:])
        if not torch.isfinite(h).all():
            raise RuntimeError("encoder output h contains NaN/Inf")

        return h


In [ ]:

# =========================================================
# 3) Eval-only window slicer + dataset + collate
# =========================================================
class EvalWindowSlicer:
    """
    Single-view eval slicer in POINTS.
    Unlike the training dataset, this is only for inference/evaluation.
    """
    def __init__(self, window_size=500, stride=250, jitter_max=0):
        self.window_size = int(window_size)
        self.stride = int(stride)
        self.jitter_max = int(jitter_max)
        if self.jitter_max != 0:
            raise ValueError("EvalWindowSlicer expects jitter_max=0 for deterministic evaluation")

    def __call__(self, chromatogram):
        rt = chromatogram["rt"]
        signal = chromatogram["signal"]

        if hasattr(rt, "cpu"):
            rt = rt.cpu().numpy()
        if hasattr(signal, "cpu"):
            signal = signal.cpu().numpy()

        rt = np.asarray(rt)
        signal = np.asarray(signal)

        Lsig = len(rt)
        windows = []
        start = 0

        while start + self.window_size <= Lsig:
            s = start
            e = s + self.window_size
            windows.append({
                "rt": rt[s:e].astype(np.float32),
                "signal": signal[s:e].astype(np.float32),
                "start": int(s),
            })
            start += self.stride

        return windows


class EvalWindowDataset(torch.utils.data.Dataset):
    """
    Evaluation-only dataset:
    - reads rt_grid / signal_grid
    - single view only
    - no augmentation
    - returns all real windows for one file
    """
    def __init__(self, npz_files, window_slicer):
        self.npz_files = [Path(p) for p in npz_files]
        self.slicer = window_slicer

    def __len__(self):
        return len(self.npz_files)

    def __getitem__(self, idx):
        npz_path = self.npz_files[idx]
        d = np.load(npz_path)

        chrom = {
            "rt": d["rt_grid"].astype(np.float32),
            "signal": d["signal_grid"].astype(np.float32),
            "chrom_name": npz_path.name,
        }

        wins = self.slicer(chrom)
        if len(wins) == 0:
            raise RuntimeError(f"No windows generated for file: {npz_path.name}")

        signal = np.stack([w["signal"] for w in wins], axis=0)   # (N, L)
        rt = np.stack([w["rt"] for w in wins], axis=0)           # (N, L)

        return {
            "signal": signal,
            "rt": rt,
            "file_name": npz_path.name,
        }


def collate_eval_window_level(batch):
    sigs, rts, names = [], [], []

    for b in batch:
        signal = b["signal"]   # (N, L)
        rt = b["rt"]           # (N, L)
        fname = b["file_name"]

        sigs.append(torch.from_numpy(signal))
        rts.append(torch.from_numpy(rt))
        names.extend([fname] * signal.shape[0])

    return {
        "signal": torch.cat(sigs, dim=0).float(),
        "rt": torch.cat(rts, dim=0).float(),
        "file_name": names,
    }


In [ ]:

# =========================================================
# 4) Helpers for labels / groups / checkpoint loading
# =========================================================
def frog_from_uhm_sample(s: str) -> str:
    return str(s).split("-")[0]


def frog_from_npz_name(fn: str) -> str:
    return str(fn).split("-")[0]


def sample_id_from_npz_name(npz_name: str) -> str:
    base = Path(npz_name).name
    m = re.search(r"-(\d+)", base)
    return m.group(1) if m else base


def sample_id_from_meta_filename(meta_filename: str) -> str:
    base = str(meta_filename)
    m = re.search(r"WF22_(\d+)", base)
    return m.group(1) if m else base


def build_npz_to_label_maps(npz_files, meta_csv):
    df = pd.read_csv(meta_csv)

    assert "UHM_sample" in df.columns
    assert "treatment" in df.columns
    assert "filename" in df.columns

    meta_by_id = {}
    for _, row in df.iterrows():
        sid = sample_id_from_meta_filename(row["filename"])
        meta_by_id[str(sid)] = row

    y_str = {}
    g_frog = {}
    uhm_sample = {}
    missing = []

    for p in npz_files:
        sid = sample_id_from_npz_name(p.name)
        if str(sid) not in meta_by_id:
            missing.append(p.name)
            continue

        r = meta_by_id[str(sid)]
        uhm = str(r["UHM_sample"])
        trt = str(r["treatment"])
        frog = frog_from_uhm_sample(uhm)

        y_str[p.name] = trt
        g_frog[p.name] = frog
        uhm_sample[p.name] = uhm

    if len(missing) > 0:
        print(f"[WARN] {len(missing)} npz files not matched to metadata. Example: {missing[:5]}")

    return y_str, g_frog, uhm_sample


def strip_prefix_any(sd, prefixes):
    for p in prefixes:
        if any(k.startswith(p) for k in sd.keys()):
            out = {k[len(p):]: v for k, v in sd.items() if k.startswith(p)}
            return out, p
    return None, None


def load_encoder_from_ckpt(
    ckpt_path: str,
    device: str,
    embed_dim=64,
    patch_size=50,
    encoder_stride=50,
    num_heads=4,
):
    ckpt = torch.load(ckpt_path, map_location="cpu")
    state = ckpt.get("state_dict", ckpt)

    load_sd, used_prefix = strip_prefix_any(
        state,
        prefixes=["enc_with_head.encoder.", "encoder.", "model.encoder."]
    )
    if load_sd is None:
        raise KeyError(f"Cannot find encoder prefix. Example keys: {list(state.keys())[:10]}")

    enc = MassSpecWindowContrastEncoder(
        embed_dim=embed_dim,
        patch_size=patch_size,
        stride=encoder_stride,
        num_heads=num_heads,
    ).to(device)

    missing, unexpected = enc.load_state_dict(load_sd, strict=False)
    enc.eval()

    print("[CKPT LOADED]")
    print("ckpt_path   :", ckpt_path)
    print("used_prefix :", used_prefix)
    print("missing     :", len(missing), missing)
    print("unexpected  :", len(unexpected), unexpected)

    return enc




def frog_aggregate_probs(p_file, y_file01, g_file, agg="mean"):
    """
    file-level probabilities -> frog-level probabilities
    """
    buckets = defaultdict(lambda: {"p": [], "y": []})

    for p, y, g in zip(p_file, y_file01, g_file):
        g = str(g)
        buckets[g]["p"].append(float(p))
        buckets[g]["y"].append(int(y))

    frogs = sorted(buckets.keys())
    p_frog = []
    y_frog = []

    for f in frogs:
        ps = buckets[f]["p"]
        ys = buckets[f]["y"]

        # 一个 frog 的标签应一致；这里保守用 majority
        y_f = int(np.mean(ys) >= 0.5)
        y_frog.append(y_f)

        if agg == "mean":
            p_frog.append(float(np.mean(ps)))
        elif agg == "median":
            p_frog.append(float(np.median(ps)))
        else:
            raise ValueError("agg must be 'mean' or 'median'")

    return np.array(p_frog, dtype=float), np.array(y_frog, dtype=int), frogs





def compute_metrics_from_probs_and_threshold(p_frog, y_frog, threshold=0.5):
    pred = (p_frog >= threshold).astype(int)

    acc = accuracy_score(y_frog, pred)
    bacc = balanced_accuracy_score(y_frog, pred)

    if len(np.unique(y_frog)) < 2:
        auroc = np.nan
        auprc = np.nan
    else:
        auroc = roc_auc_score(y_frog, p_frog)
        auprc = average_precision_score(y_frog, p_frog)

    return {
        "ACC": float(acc),
        "Balanced_ACC": float(bacc),
        "AUROC": float(auroc) if not np.isnan(auroc) else np.nan,
        "AUPRC": float(auprc) if not np.isnan(auprc) else np.nan,
    }


def choose_best_threshold(y_true, p_prob, metric="balanced_acc"):
    """
    在 inner OOF frog-level probabilities 上选阈值。
    metric:
      - "balanced_acc"  : 推荐
      - "acc"
    """
    y_true = np.asarray(y_true).astype(int)
    p_prob = np.asarray(p_prob).astype(float)

    # 用所有概率值本身 + 0/1 端点构造候选阈值
    cand = np.unique(np.concatenate([
        np.array([0.0, 0.5, 1.0]),
        p_prob
    ]))

    best_thr = 0.5
    best_score = -np.inf

    for thr in cand:
        pred = (p_prob >= thr).astype(int)

        if metric == "balanced_acc":
            score = balanced_accuracy_score(y_true, pred)
        elif metric == "acc":
            score = accuracy_score(y_true, pred)
        else:
            raise ValueError("metric must be 'balanced_acc' or 'acc'")

        # 平分时偏向更接近 0.5 的阈值，避免极端阈值
        if (score > best_score) or (score == best_score and abs(thr - 0.5) < abs(best_thr - 0.5)):
            best_score = score
            best_thr = float(thr)

    return best_thr, float(best_score)


def inner_logo_select_threshold(X_train, y_train, g_train, C=40.0, frog_prob_agg="mean", metric="balanced_acc"):
    """
    在 outer-train frogs 内再做一层 LOGO：
    1) 生成 inner OOF file probabilities
    2) 聚合成 frog-level probabilities
    3) 选最佳 threshold
    """
    clf_factory = lambda: make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            max_iter=5000,
            class_weight="balanced",
            solver="lbfgs",
        ),
    )

    logo_inner = LeaveOneGroupOut()
    p_file_inner_oof = np.full(len(y_train), np.nan, dtype=float)

    skipped_inner = 0

    for tr_in, va_in in logo_inner.split(X_train, y_train, g_train):
        y_tr_in = y_train[tr_in]

        # inner-train 若只有单类，则跳过
        if len(np.unique(y_tr_in)) < 2:
            skipped_inner += 1
            continue

        clf = clf_factory()
        clf.fit(X_train[tr_in], y_tr_in)
        p_file_inner_oof[va_in] = clf.predict_proba(X_train[va_in])[:, 1]

    valid = ~np.isnan(p_file_inner_oof)
    if valid.sum() == 0:
        # 极端情况下退回默认阈值
        return 0.5, np.nan, skipped_inner, None, None

    p_frog_inner, y_frog_inner, frogs_inner = frog_aggregate_probs(
        p_file_inner_oof[valid],
        y_train[valid],
        g_train[valid],
        agg=frog_prob_agg,
    )

    best_thr, best_score = choose_best_threshold(
        y_true=y_frog_inner,
        p_prob=p_frog_inner,
        metric=metric,
    )

    inner_frog_df = pd.DataFrame({
        "frog_id": frogs_inner,
        "y_true": y_frog_inner,
        "prob_frog": p_frog_inner,
    }).sort_values(["y_true", "prob_frog"], ascending=[False, False]).reset_index(drop=True)

    return best_thr, best_score, skipped_inner, inner_frog_df, p_file_inner_oof




def logo_oof_logreg_frog_level_with_nested_threshold(
    E_file,
    y_file_str,
    g_file,
    pos_label,
    neg_label,
    C=40.0,
    frog_prob_agg="mean",
    threshold_metric="balanced_acc",   # or "acc"
):
    """
    outer LOGO:
      - outer train frogs 上做 inner LOGO 选 threshold
      - outer test frog 用该 threshold 判别
    输出：
      - default 0.5 结果
      - tuned threshold 结果
    """
    keep = np.isin(y_file_str, [pos_label, neg_label])

    X = np.asarray(E_file[keep], dtype=float)
    y01 = (y_file_str[keep] == pos_label).astype(int)
    g = np.asarray(g_file[keep], dtype=str)

    print("\n" + "=" * 70)
    print(f"OOF LOGO + nested threshold tuning | POS={pos_label} vs NEG={neg_label}")
    print(f"n_files={len(y01)} | n_frogs={len(np.unique(g))} | prevalence={y01.mean():.3f}")

    clf_factory = lambda: make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            max_iter=5000,
            class_weight="balanced",
            solver="lbfgs",
        ),
    )

    logo = LeaveOneGroupOut()

    p_file_oof = np.full(len(y01), np.nan, dtype=float)
    outer_thresholds = {}
    skipped_outer = 0

    fold_rows = []
    frog_rows = []

    for fold_idx, (tr, te) in enumerate(logo.split(X, y01, g), start=1):
        test_frog = np.unique(g[te])
        assert len(test_frog) == 1
        test_frog = str(test_frog[0])

        X_tr, y_tr, g_tr = X[tr], y01[tr], g[tr]
        X_te, y_te, g_te = X[te], y01[te], g[te]

        if len(np.unique(y_tr)) < 2:
            skipped_outer += 1
            continue

        # inner LOGO 选 threshold
        best_thr, best_score, skipped_inner, inner_frog_df, _ = inner_logo_select_threshold(
            X_train=X_tr,
            y_train=y_tr,
            g_train=g_tr,
            C=C,
            frog_prob_agg=frog_prob_agg,
            metric=threshold_metric,
        )

        # 用整个 outer-train 训练，再预测 outer-test
        clf = clf_factory()
        clf.fit(X_tr, y_tr)
        p_te = clf.predict_proba(X_te)[:, 1]
        p_file_oof[te] = p_te

        outer_thresholds[test_frog] = best_thr

        # 当前 outer test frog 的 frog-level prob
        p_frog_te, y_frog_te, frogs_te = frog_aggregate_probs(
            p_te, y_te, g_te, agg=frog_prob_agg
        )
        assert len(frogs_te) == 1

        frog_prob = float(p_frog_te[0])
        frog_true = int(y_frog_te[0])

        frog_rows.append({
            "frog_id": test_frog,
            "y_true": frog_true,
            "prob_frog": frog_prob,
            "pred_default_0.5": int(frog_prob >= 0.5),
            "best_threshold": float(best_thr),
            "pred_tuned": int(frog_prob >= best_thr),
            "inner_best_score": float(best_score) if not np.isnan(best_score) else np.nan,
        })

        fold_rows.append({
            "fold": fold_idx,
            "test_frog": test_frog,
            "n_test_files": int(len(te)),
            "test_label": int(np.mean(y_te) >= 0.5),
            "prob_mean_file": float(np.mean(p_te)),
            "prob_median_file": float(np.median(p_te)),
            "best_threshold": float(best_thr),
            "inner_best_score": float(best_score) if not np.isnan(best_score) else np.nan,
            "skipped_inner_folds": int(skipped_inner),
        })

    if np.isnan(p_file_oof).any():
        bad_idx = np.where(np.isnan(p_file_oof))[0]
        raise RuntimeError(f"Some OOF probabilities are NaN. bad_idx[:10]={bad_idx[:10]}")

    # frog-level OOF probs（统一汇总）
    p_frog, y_frog, frogs = frog_aggregate_probs(
        p_file_oof, y01, g, agg=frog_prob_agg
    )

    # 默认 0.5
    default_metrics = compute_metrics_from_probs_and_threshold(
        p_frog=p_frog,
        y_frog=y_frog,
        threshold=0.5,
    )

    # tuned threshold：每个 frog 用对应 outer fold 选到的 threshold
    frog_df = pd.DataFrame(frog_rows).sort_values(
        ["y_true", "prob_frog"], ascending=[False, False]
    ).reset_index(drop=True)

    y_frog_tuned = frog_df["y_true"].values.astype(int)
    pred_tuned = frog_df["pred_tuned"].values.astype(int)
    p_frog_tuned = frog_df["prob_frog"].values.astype(float)

    tuned_acc = accuracy_score(y_frog_tuned, pred_tuned)
    tuned_bacc = balanced_accuracy_score(y_frog_tuned, pred_tuned)

    tuned_metrics = {
        "ACC": float(tuned_acc),
        "Balanced_ACC": float(tuned_bacc),
        "AUROC": float(roc_auc_score(y_frog_tuned, p_frog_tuned)),
        "AUPRC": float(average_precision_score(y_frog_tuned, p_frog_tuned)),
    }

    fold_df = pd.DataFrame(fold_rows)

    summary_df = pd.DataFrame([
        {
            "mode": "OOF_LOGO_LR_default",
            "ACC": default_metrics["ACC"],
            "Balanced_ACC": default_metrics["Balanced_ACC"],
            "AUROC": default_metrics["AUROC"],
            "AUPRC": default_metrics["AUPRC"],
            "n_frogs": len(y_frog),
            "n_pos_frogs": int(y_frog.sum()),
            "n_neg_frogs": int((1 - y_frog).sum()),
            "skipped_folds": float(skipped_outer),
            "threshold_metric": threshold_metric,
        },
        {
            "mode": "OOF_LOGO_LR_tuned",
            "ACC": tuned_metrics["ACC"],
            "Balanced_ACC": tuned_metrics["Balanced_ACC"],
            "AUROC": tuned_metrics["AUROC"],
            "AUPRC": tuned_metrics["AUPRC"],
            "n_frogs": len(y_frog_tuned),
            "n_pos_frogs": int(y_frog_tuned.sum()),
            "n_neg_frogs": int((1 - y_frog_tuned).sum()),
            "skipped_folds": float(skipped_outer),
            "threshold_metric": threshold_metric,
        },
    ])

    print("\n[OOF LOGO LR | default threshold = 0.5]")
    print(default_metrics)

    print("\n[OOF LOGO LR | tuned threshold]")
    print(tuned_metrics)

    return {
        "summary_df": summary_df,
        "frog_df": frog_df,
        "fold_df": fold_df,
        "p_file_oof": p_file_oof,
        "y_file01": y01,
        "g_file": g,
        "outer_thresholds": outer_thresholds,
    }

In [ ]:

# =========================================================
# 5) Extract file embeddings
# =========================================================
@torch.no_grad()
def extract_E_file_from_encoder(encoder, loader, device, y_file_str_map):
    encoder = encoder.to(device).eval()

    Z_sum = defaultdict(lambda: None)
    Z_cnt = defaultdict(int)
    y_str = {}

    for batch in loader:
        signal = batch["signal"].to(device)   # (Bwin, L)
        rt = batch["rt"].to(device)           # (Bwin, L)
        names = list(batch["file_name"])      # len = Bwin

        z = encoder(signal, rt)               # (Bwin, D)
        z = z.detach().cpu().numpy().astype(np.float64)

        for zi, fn in zip(z, names):
            fn = str(fn)
            if fn not in y_file_str_map:
                continue

            if Z_sum[fn] is None:
                Z_sum[fn] = zi.copy()
            else:
                Z_sum[fn] += zi

            Z_cnt[fn] += 1
            y_str[fn] = y_file_str_map[fn]

    file_names = sorted(Z_sum.keys())
    E_file = np.stack([Z_sum[n] / max(1, Z_cnt[n]) for n in file_names], axis=0)
    y_file_str = np.array([y_str[n] for n in file_names], dtype=object)
    g_file = np.array([frog_from_npz_name(n) for n in file_names], dtype=str)

    print("\n[WINDOW COUNT PER FILE]")
    for fn in file_names[:10]:
        print(fn, "->", Z_cnt[fn])
    if len(file_names) > 10:
        print("... total files:", len(file_names))

    return E_file, y_file_str, g_file, file_names


In [ ]:

# =========================================================
# 6) Frog-level evaluation: in-sample + OOF, LR + LDA
# =========================================================
def aggregate_by_group(values, groups, mode="mean"):
    buckets = defaultdict(list)
    for v, g in zip(values, groups):
        buckets[g].append(float(v))

    group_ids = list(buckets.keys())

    if mode == "mean":
        agg = np.array([np.mean(buckets[g]) for g in group_ids], dtype=float)
    elif mode == "median":
        agg = np.array([np.median(buckets[g]) for g in group_ids], dtype=float)
    else:
        raise ValueError(mode)

    return agg, np.array(group_ids, dtype=object)


def frog_metrics_from_file_probs(P_file, Y_file, G_file, threshold=0.5, frog_agg="mean"):
    P_frog, frogs = aggregate_by_group(P_file, G_file, mode=frog_agg)
    Y_frog_mean, _ = aggregate_by_group(Y_file, G_file, mode="mean")
    Y_frog = (Y_frog_mean >= 0.5).astype(int)

    pred_frog = (P_frog >= threshold).astype(int)
    frog_acc = accuracy_score(Y_frog, pred_frog)

    if len(np.unique(Y_frog)) < 2:
        frog_auroc = np.nan
        frog_auprc = np.nan
    else:
        frog_auroc = roc_auc_score(Y_frog, P_frog)
        frog_auprc = average_precision_score(Y_frog, P_frog)

    frog_df = pd.DataFrame({
        "frog_id": frogs,
        "y_true": Y_frog,
        "prob_frog": P_frog,
        "pred_0.5": pred_frog,
    }).sort_values(["y_true", "prob_frog"], ascending=[False, False]).reset_index(drop=True)

    metrics = {
        "ACC": float(frog_acc),
        "AUROC": float(frog_auroc),
        "AUPRC": float(frog_auprc),
        "n_frogs": int(len(Y_frog)),
        "n_pos_frogs": int(Y_frog.sum()),
        "n_neg_frogs": int((1 - Y_frog).sum()),
        "frog_df": frog_df,
    }
    return metrics


def make_lr(C=10.0):
    return make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=C,
            max_iter=5000,
            class_weight="balanced",
            solver="lbfgs",
        ),
    )


def make_lda(shrinkage="auto"):
    return LinearDiscriminantAnalysis(
        solver="lsqr",
        shrinkage=shrinkage,
    )


def evaluate_task_all_modes(
    E,
    y_str,
    groups_frog,
    pos_class,
    neg_class="control",
    C=10.0,
    threshold=0.5,
    frog_agg="mean",
    lda_shrinkage="auto",
):
    E = np.asarray(E)
    y_str = np.asarray(y_str, dtype=object)
    g = np.asarray(groups_frog, dtype=object)

    keep = np.isin(y_str, [pos_class, neg_class])
    X, y2_str, g2 = E[keep], y_str[keep], g[keep]

    if X.shape[0] == 0:
        raise ValueError(f"No samples for {pos_class} vs {neg_class}")

    y = (y2_str == pos_class).astype(int)
    logo = LeaveOneGroupOut()

    # -------------------------
    # in-sample LR
    # -------------------------
    clf_lr_in = make_lr(C=C)
    clf_lr_in.fit(X, y)
    P_file_lr_in = clf_lr_in.predict_proba(X)[:, 1]
    res_lr_in = frog_metrics_from_file_probs(
        P_file_lr_in, y, g2, threshold=threshold, frog_agg=frog_agg
    )

    # -------------------------
    # in-sample LDA
    # -------------------------
    clf_lda_in = make_lda(shrinkage=lda_shrinkage)
    clf_lda_in.fit(X, y)
    P_file_lda_in = clf_lda_in.predict_proba(X)[:, 1]
    res_lda_in = frog_metrics_from_file_probs(
        P_file_lda_in, y, g2, threshold=threshold, frog_agg=frog_agg
    )

    # -------------------------
    # OOF LOGO LR
    # -------------------------
    P_file_lr_oof_all, Y_file_lr_oof_all, G_file_lr_oof_all = [], [], []
    fold_rows_lr = []
    n_skip_lr = 0

    for fold_idx, (tr, te) in enumerate(logo.split(X, y, g2), start=1):
        test_frog = np.unique(g2[te])
        assert len(test_frog) == 1
        test_frog = test_frog[0]

        if len(np.unique(y[tr])) < 2:
            n_skip_lr += 1
            continue

        clf = make_lr(C=C)
        clf.fit(X[tr], y[tr])

        p = clf.predict_proba(X[te])[:, 1]
        P_file_lr_oof_all.append(p)
        Y_file_lr_oof_all.append(y[te])
        G_file_lr_oof_all.append(g2[te])

        fold_rows_lr.append({
            "fold": int(fold_idx),
            "test_frog": str(test_frog),
            "n_test_files": int(len(te)),
            "test_label": int(np.mean(y[te]) >= 0.5),
            "prob_mean_file": float(np.mean(p)),
            "prob_median_file": float(np.median(p)),
        })

    if len(P_file_lr_oof_all) == 0:
        raise ValueError(f"All LR folds skipped. skipped={n_skip_lr}")

    P_file_lr_oof = np.concatenate(P_file_lr_oof_all)
    Y_file_lr_oof = np.concatenate(Y_file_lr_oof_all)
    G_file_lr_oof = np.concatenate(G_file_lr_oof_all)

    res_lr_oof = frog_metrics_from_file_probs(
        P_file_lr_oof, Y_file_lr_oof, G_file_lr_oof, threshold=threshold, frog_agg=frog_agg
    )
    res_lr_oof["skipped_folds"] = int(n_skip_lr)
    res_lr_oof["fold_df"] = pd.DataFrame(fold_rows_lr)

    # -------------------------
    # OOF LOGO LDA
    # -------------------------
    P_file_lda_oof_all, Y_file_lda_oof_all, G_file_lda_oof_all = [], [], []
    fold_rows_lda = []
    n_skip_lda = 0

    for fold_idx, (tr, te) in enumerate(logo.split(X, y, g2), start=1):
        test_frog = np.unique(g2[te])
        assert len(test_frog) == 1
        test_frog = test_frog[0]

        if len(np.unique(y[tr])) < 2:
            n_skip_lda += 1
            continue

        clf = make_lda(shrinkage=lda_shrinkage)
        clf.fit(X[tr], y[tr])

        p = clf.predict_proba(X[te])[:, 1]
        P_file_lda_oof_all.append(p)
        Y_file_lda_oof_all.append(y[te])
        G_file_lda_oof_all.append(g2[te])

        fold_rows_lda.append({
            "fold": int(fold_idx),
            "test_frog": str(test_frog),
            "n_test_files": int(len(te)),
            "test_label": int(np.mean(y[te]) >= 0.5),
            "prob_mean_file": float(np.mean(p)),
            "prob_median_file": float(np.median(p)),
        })

    if len(P_file_lda_oof_all) == 0:
        raise ValueError(f"All LDA folds skipped. skipped={n_skip_lda}")

    P_file_lda_oof = np.concatenate(P_file_lda_oof_all)
    Y_file_lda_oof = np.concatenate(Y_file_lda_oof_all)
    G_file_lda_oof = np.concatenate(G_file_lda_oof_all)

    res_lda_oof = frog_metrics_from_file_probs(
        P_file_lda_oof, Y_file_lda_oof, G_file_lda_oof, threshold=threshold, frog_agg=frog_agg
    )
    res_lda_oof["skipped_folds"] = int(n_skip_lda)
    res_lda_oof["fold_df"] = pd.DataFrame(fold_rows_lda)

    summary_df = pd.DataFrame([
        {
            "mode": "in_sample_LR",
            "ACC": res_lr_in["ACC"],
            "AUROC": res_lr_in["AUROC"],
            "AUPRC": res_lr_in["AUPRC"],
            "n_frogs": res_lr_in["n_frogs"],
            "n_pos_frogs": res_lr_in["n_pos_frogs"],
            "n_neg_frogs": res_lr_in["n_neg_frogs"],
            "skipped_folds": np.nan,
        },
        {
            "mode": "in_sample_LDA",
            "ACC": res_lda_in["ACC"],
            "AUROC": res_lda_in["AUROC"],
            "AUPRC": res_lda_in["AUPRC"],
            "n_frogs": res_lda_in["n_frogs"],
            "n_pos_frogs": res_lda_in["n_pos_frogs"],
            "n_neg_frogs": res_lda_in["n_neg_frogs"],
            "skipped_folds": np.nan,
        },
        {
            "mode": "OOF_LOGO_LR",
            "ACC": res_lr_oof["ACC"],
            "AUROC": res_lr_oof["AUROC"],
            "AUPRC": res_lr_oof["AUPRC"],
            "n_frogs": res_lr_oof["n_frogs"],
            "n_pos_frogs": res_lr_oof["n_pos_frogs"],
            "n_neg_frogs": res_lr_oof["n_neg_frogs"],
            "skipped_folds": res_lr_oof["skipped_folds"],
        },
        {
            "mode": "OOF_LOGO_LDA",
            "ACC": res_lda_oof["ACC"],
            "AUROC": res_lda_oof["AUROC"],
            "AUPRC": res_lda_oof["AUPRC"],
            "n_frogs": res_lda_oof["n_frogs"],
            "n_pos_frogs": res_lda_oof["n_pos_frogs"],
            "n_neg_frogs": res_lda_oof["n_neg_frogs"],
            "skipped_folds": res_lda_oof["skipped_folds"],
        },
    ])

    return {
        "summary_df": summary_df,
        "in_sample_LR": res_lr_in,
        "in_sample_LDA": res_lda_in,
        "OOF_LOGO_LR": res_lr_oof,
        "OOF_LOGO_LDA": res_lda_oof,
    }


def parse_seed_step_from_ckpt_name(ckpt_name: str):
    ckpt_name = Path(ckpt_name).name
    m_seed = re.search(r"seed(\d+)", ckpt_name)
    m_step = re.search(r"step(\d+)", ckpt_name)
    seed = int(m_seed.group(1)) if m_seed else np.nan
    step = int(m_step.group(1)) if m_step else np.nan
    return seed, step


In [ ]:
# =========================================================
# 7) Run all checkpoints + save FINAL CSVs
# =========================================================
EVAL_BATCH_SIZE_FILES = 16
THRESHOLD = 0.5   # only for non-tuned ACC in in-sample / plain LOGO

npz_dir = Path(NPZ_DIR)
npz_files = sorted(npz_dir.glob("*.npz"))
print("Total npz:", len(npz_files))

y_file_str_map, g_frog_map, uhm_sample_map = build_npz_to_label_maps(npz_files, META_CSV)

slicer = EvalWindowSlicer(
    window_size=WINDOW_SIZE,
    stride=STRIDE,
    jitter_max=JITTER_MAX,
)

eval_ds = EvalWindowDataset(
    npz_files=npz_files,
    window_slicer=slicer,
)

eval_loader = DataLoader(
    eval_ds,
    batch_size=EVAL_BATCH_SIZE_FILES,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_eval_window_level,
    drop_last=False,
)

ckpt_dir = Path(CKPT_DIR)
ckpt_paths = sorted(ckpt_dir.glob("*.ckpt"))
if len(ckpt_paths) == 0:
    raise FileNotFoundError(f"No ckpt found in: {ckpt_dir}")

print("\n[CKPTS FOUND]")
for p in ckpt_paths:
    print(" -", p.name)

output_dir = Path(OUTPUT_DIR)
output_dir.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------
# containers
# ---------------------------------------------------------
summary_rows = []                 # final wide summary: one row per ckpt
plot_ready_rows = []             # long-form for plotting
frog_rows_all = []               # frog-level predictions from LR/LDA
fold_rows_all = []               # fold-level details from LR/LDA
nested_summary_rows = []         # tuned threshold summary
nested_frog_rows_all = []        # tuned threshold frog details
nested_fold_rows_all = []        # tuned threshold fold details


def get_mode_metric(df_summary, mode_name, metric_name):
    sub = df_summary[df_summary["mode"] == mode_name]
    if len(sub) == 0:
        return np.nan
    return sub.iloc[0][metric_name]


for i, ckpt_path in enumerate(ckpt_paths, start=1):
    ckpt_name = ckpt_path.name
    seed, step = parse_seed_step_from_ckpt_name(ckpt_name)

    print("\n" + "=" * 90)
    print(f"[{i}/{len(ckpt_paths)}] Evaluating {ckpt_name}")
    print("=" * 90)

    # -----------------------------------------------------
    # load encoder and extract file embeddings
    # -----------------------------------------------------
    encoder = load_encoder_from_ckpt(str(ckpt_path), DEVICE)
    E_file, y_file_str, g_file, file_names = extract_E_file_from_encoder(
        encoder, eval_loader, DEVICE, y_file_str_map
    )

    print("\n[DATA SUMMARY]")
    print("E_file shape:", E_file.shape)
    print("n_files     :", len(file_names))
    print("classes     :", sorted(set(y_file_str.tolist())))
    print("n_frogs     :", len(set(g_file.tolist())))

    # -----------------------------------------------------
    # plain evaluation: LR + LDA, in-sample + LOGO
    # -----------------------------------------------------
    results = evaluate_task_all_modes(
        E=E_file,
        y_str=y_file_str,
        groups_frog=g_file,
        pos_class=POS_LABEL,
        neg_class=NEG_LABEL,
        C=LOGREG_C,
        threshold=THRESHOLD,
        frog_agg=FROG_PROB_AGG,
        lda_shrinkage=LDA_SHRINKAGE,
    )

    df_sum = results["summary_df"].copy()

    # -----------------------------------------------------
    # nested threshold tuning for ACC / balanced ACC
    # -----------------------------------------------------
    nested_thr_res = logo_oof_logreg_frog_level_with_nested_threshold(
        E_file=E_file,
        y_file_str=y_file_str,
        g_file=g_file,
        pos_label=POS_LABEL,
        neg_label=NEG_LABEL,
        C=LOGREG_C,
        frog_prob_agg=FROG_PROB_AGG,
        threshold_metric="balanced_acc",
    )

    nested_sum = nested_thr_res["summary_df"].copy()

    # =====================================================
    # A) final one-row summary per ckpt
    # =====================================================
    row = {
        "ckpt_name": ckpt_name,
        "ckpt_path": str(ckpt_path),
        "seed": seed,
        "step": step,
        "task": f"{POS_LABEL}_vs_{NEG_LABEL}",
        "pos_label": POS_LABEL,
        "neg_label": NEG_LABEL,
        "frog_agg": FROG_PROB_AGG,
        "C": LOGREG_C,
        "lda_shrinkage": LDA_SHRINKAGE,

        # -------------------------
        # LR in-sample
        # -------------------------
        "ACC_LR_insample":   get_mode_metric(df_sum, "in_sample_LR", "ACC"),
        "AUROC_LR_insample": get_mode_metric(df_sum, "in_sample_LR", "AUROC"),
        "AUPRC_LR_insample": get_mode_metric(df_sum, "in_sample_LR", "AUPRC"),

        # -------------------------
        # LDA in-sample
        # -------------------------
        "ACC_LDA_insample":   get_mode_metric(df_sum, "in_sample_LDA", "ACC"),
        "AUROC_LDA_insample": get_mode_metric(df_sum, "in_sample_LDA", "AUROC"),
        "AUPRC_LDA_insample": get_mode_metric(df_sum, "in_sample_LDA", "AUPRC"),

        # -------------------------
        # LR LOGO
        # -------------------------
        "ACC_LR_logo":   get_mode_metric(df_sum, "OOF_LOGO_LR", "ACC"),
        "AUROC_LR_logo": get_mode_metric(df_sum, "OOF_LOGO_LR", "AUROC"),
        "AUPRC_LR_logo": get_mode_metric(df_sum, "OOF_LOGO_LR", "AUPRC"),
        "skipped_folds_LR_logo": get_mode_metric(df_sum, "OOF_LOGO_LR", "skipped_folds"),

        # -------------------------
        # LDA LOGO
        # -------------------------
        "ACC_LDA_logo":   get_mode_metric(df_sum, "OOF_LOGO_LDA", "ACC"),
        "AUROC_LDA_logo": get_mode_metric(df_sum, "OOF_LOGO_LDA", "AUROC"),
        "AUPRC_LDA_logo": get_mode_metric(df_sum, "OOF_LOGO_LDA", "AUPRC"),
        "skipped_folds_LDA_logo": get_mode_metric(df_sum, "OOF_LOGO_LDA", "skipped_folds"),

        # -------------------------
        # tuned threshold LOGO (LogReg only)
        # -------------------------
        "ACC_tuned_logo": None,
        "Balanced_ACC_tuned_logo": None,
        "AUROC_tuned_logo": None,
        "AUPRC_tuned_logo": None,
        "threshold_metric": "balanced_acc",
    }

    # nested summary: assume only one row
    if len(nested_sum) > 0:
        row["ACC_tuned_logo"] = nested_sum.iloc[0].get("ACC_tuned", np.nan)
        row["Balanced_ACC_tuned_logo"] = nested_sum.iloc[0].get("Balanced_ACC_tuned", np.nan)
        row["AUROC_tuned_logo"] = nested_sum.iloc[0].get("AUROC", np.nan)
        row["AUPRC_tuned_logo"] = nested_sum.iloc[0].get("AUPRC", np.nan)

    # counts
    row["n_frogs"] = get_mode_metric(df_sum, "OOF_LOGO_LR", "n_frogs")
    row["n_pos_frogs"] = get_mode_metric(df_sum, "OOF_LOGO_LR", "n_pos_frogs")
    row["n_neg_frogs"] = get_mode_metric(df_sum, "OOF_LOGO_LR", "n_neg_frogs")

    summary_rows.append(row)

    # =====================================================
    # B) plot-ready long table
    # =====================================================
    plot_specs = [
        ("LogReg", "in_sample", "AUROC", row["AUROC_LR_insample"]),
        ("LogReg", "in_sample", "AUPRC", row["AUPRC_LR_insample"]),
        ("LogReg", "in_sample", "ACC",   row["ACC_LR_insample"]),

        ("LDA",    "in_sample", "AUROC", row["AUROC_LDA_insample"]),
        ("LDA",    "in_sample", "AUPRC", row["AUPRC_LDA_insample"]),
        ("LDA",    "in_sample", "ACC",   row["ACC_LDA_insample"]),

        ("LogReg", "LOGO", "AUROC", row["AUROC_LR_logo"]),
        ("LogReg", "LOGO", "AUPRC", row["AUPRC_LR_logo"]),
        ("LogReg", "LOGO", "ACC",   row["ACC_LR_logo"]),

        ("LDA",    "LOGO", "AUROC", row["AUROC_LDA_logo"]),
        ("LDA",    "LOGO", "AUPRC", row["AUPRC_LDA_logo"]),
        ("LDA",    "LOGO", "ACC",   row["ACC_LDA_logo"]),

        ("LogReg", "LOGO_tuned", "ACC",          row["ACC_tuned_logo"]),
        ("LogReg", "LOGO_tuned", "Balanced_ACC", row["Balanced_ACC_tuned_logo"]),
    ]

    for probe, setting, metric, value in plot_specs:
        plot_ready_rows.append({
            "ckpt_name": ckpt_name,
            "ckpt_path": str(ckpt_path),
            "seed": seed,
            "step": step,
            "task": f"{POS_LABEL}_vs_{NEG_LABEL}",
            "probe": probe,
            "setting": setting,
            "metric": metric,
            "value": value,
        })

    # =====================================================
    # C) frog-level details
    # =====================================================
    for mode_key in ["in_sample_LR", "in_sample_LDA", "OOF_LOGO_LR", "OOF_LOGO_LDA"]:
        frog_df = results[mode_key]["frog_df"].copy()
        frog_df.insert(0, "mode", mode_key)
        frog_df.insert(0, "task", f"{POS_LABEL}_vs_{NEG_LABEL}")
        frog_df.insert(0, "step", step)
        frog_df.insert(0, "seed", seed)
        frog_df.insert(0, "ckpt_name", ckpt_name)
        frog_df.insert(0, "ckpt_path", str(ckpt_path))
        frog_rows_all.append(frog_df)

        if "fold_df" in results[mode_key]:
            fold_df = results[mode_key]["fold_df"].copy()
            fold_df.insert(0, "mode", mode_key)
            fold_df.insert(0, "task", f"{POS_LABEL}_vs_{NEG_LABEL}")
            fold_df.insert(0, "step", step)
            fold_df.insert(0, "seed", seed)
            fold_df.insert(0, "ckpt_name", ckpt_name)
            fold_df.insert(0, "ckpt_path", str(ckpt_path))
            fold_rows_all.append(fold_df)

    # =====================================================
    # D) nested-threshold summary + details
    # =====================================================
    if len(nested_sum) > 0:
        nested_sum2 = nested_sum.copy()
        nested_sum2.insert(0, "ckpt_name", ckpt_name)
        nested_sum2.insert(1, "ckpt_path", str(ckpt_path))
        nested_sum2.insert(2, "seed", seed)
        nested_sum2.insert(3, "step", step)
        nested_sum2.insert(4, "task", f"{POS_LABEL}_vs_{NEG_LABEL}")
        nested_sum2["frog_agg"] = FROG_PROB_AGG
        nested_sum2["C"] = LOGREG_C
        nested_sum2["threshold_metric"] = "balanced_acc"
        nested_summary_rows.append(nested_sum2)

    nested_frog_df = nested_thr_res["frog_df"].copy()
    nested_frog_df.insert(0, "mode", "OOF_LOGO_LR_nested_threshold")
    nested_frog_df.insert(0, "task", f"{POS_LABEL}_vs_{NEG_LABEL}")
    nested_frog_df.insert(0, "step", step)
    nested_frog_df.insert(0, "seed", seed)
    nested_frog_df.insert(0, "ckpt_name", ckpt_name)
    nested_frog_df.insert(0, "ckpt_path", str(ckpt_path))
    nested_frog_rows_all.append(nested_frog_df)

    nested_fold_df = nested_thr_res["fold_df"].copy()
    nested_fold_df.insert(0, "mode", "OOF_LOGO_LR_nested_threshold")
    nested_fold_df.insert(0, "task", f"{POS_LABEL}_vs_{NEG_LABEL}")
    nested_fold_df.insert(0, "step", step)
    nested_fold_df.insert(0, "seed", seed)
    nested_fold_df.insert(0, "ckpt_name", ckpt_name)
    nested_fold_df.insert(0, "ckpt_path", str(ckpt_path))
    nested_fold_rows_all.append(nested_fold_df)


# =========================================================
# concat
# =========================================================
summary_df_all = pd.DataFrame(summary_rows).sort_values(["seed", "step"]).reset_index(drop=True)
plot_ready_df_all = pd.DataFrame(plot_ready_rows).sort_values(["probe", "setting", "metric", "seed"]).reset_index(drop=True)

frog_df_all = pd.concat(frog_rows_all, axis=0, ignore_index=True) if len(frog_rows_all) else pd.DataFrame()
fold_df_all = pd.concat(fold_rows_all, axis=0, ignore_index=True) if len(fold_rows_all) else pd.DataFrame()

nested_summary_df_all = pd.concat(nested_summary_rows, axis=0, ignore_index=True) if len(nested_summary_rows) else pd.DataFrame()
nested_frog_df_all = pd.concat(nested_frog_rows_all, axis=0, ignore_index=True) if len(nested_frog_rows_all) else pd.DataFrame()
nested_fold_df_all = pd.concat(nested_fold_rows_all, axis=0, ignore_index=True) if len(nested_fold_rows_all) else pd.DataFrame()


# =========================================================
# save
# =========================================================
summary_csv = output_dir / f"all_ckpt_summary__{POS_LABEL}_vs_{NEG_LABEL}.csv"
plot_ready_csv = output_dir / f"all_ckpt_plot_ready__{POS_LABEL}_vs_{NEG_LABEL}.csv"
frog_csv = output_dir / f"all_ckpt_frog_details__{POS_LABEL}_vs_{NEG_LABEL}.csv"
fold_csv = output_dir / f"all_ckpt_fold_details__{POS_LABEL}_vs_{NEG_LABEL}.csv"
nested_summary_csv = output_dir / f"all_ckpt_nested_threshold_summary__{POS_LABEL}_vs_{NEG_LABEL}.csv"
nested_frog_csv = output_dir / f"all_ckpt_nested_threshold_frog_details__{POS_LABEL}_vs_{NEG_LABEL}.csv"
nested_fold_csv = output_dir / f"all_ckpt_nested_threshold_fold_details__{POS_LABEL}_vs_{NEG_LABEL}.csv"

summary_df_all.to_csv(summary_csv, index=False)
plot_ready_df_all.to_csv(plot_ready_csv, index=False)
frog_df_all.to_csv(frog_csv, index=False)
fold_df_all.to_csv(fold_csv, index=False)
nested_summary_df_all.to_csv(nested_summary_csv, index=False)
nested_frog_df_all.to_csv(nested_frog_csv, index=False)
nested_fold_df_all.to_csv(nested_fold_csv, index=False)

print("\n[SAVED]")
print(summary_csv)
print(plot_ready_csv)
print(frog_csv)
print(fold_csv)
print(nested_summary_csv)
print(nested_frog_csv)
print(nested_fold_csv)

print("\n[SUMMARY PREVIEW]")
display(summary_df_all)